# Phase 2: Causal Modeling with DoWhy (MIND-large)

This notebook implements the DoWhy causal modeling framework based on the Phase 1 extracted data (`data/scm_train.parquet`) built from MIND-large.

## Tuning for MIND-large
- MIND-large produces significantly more SCM rows, yielding more precise ATE estimates.
- PCA explained variance may differ with more diverse user/item embeddings.
- The causal structure and refutation checks remain identical to MIND-small.

In [ ]:
import warnings
import logging
import pandas as pd

import sys
from pathlib import Path
_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.causal_model.graph import build_causal_graph_gml
from src.causal_model.model import create_causal_model, identify_effect, estimate_ate_ipw, estimate_ate_linear, positivity_check
from src.causal_model.refutation import run_refutations
from src.causal_model.cdi import compute_cdi_batch
import dowhy
print(f'DoWhy Version: {dowhy.__version__}')

warnings.filterwarnings('ignore')
logging.getLogger('dowhy').setLevel(logging.WARNING)
logging.getLogger('statsmodels').setLevel(logging.WARNING)


## 2. Load Data & Define Graph
Load `scm_train.parquet` (from MIND-large pipeline) and define the tiered GML causal graph.

In [ ]:
data_path = str(_root / 'data' / 'scm_train.parquet')
try:
    df = pd.read_parquet(data_path)
    print(f'Loaded Phase 1 data: {len(df)} records.')
except FileNotFoundError:
    print(f'Warning: {data_path} not found. Please run Phase 1 data pipeline first.')
    df = pd.DataFrame()

u_pca_cols = [col for col in df.columns if col.startswith('U_pca_')] if not df.empty else ['U_pca_0']
causal_graph_gml = build_causal_graph_gml(u_pca_cols)


In [ ]:
if not df.empty:
    common_causes = [col for col in df.columns if col.startswith('U_pca_')]
    common_causes += ['U_dwell_mean', 'I_category', 'I_sentiment']
    model = create_causal_model(
        df=df, treatment='A', outcome='Y_diversity',
        graph_gml=causal_graph_gml, common_causes=common_causes,
    )
    try:
        model.view_model(layout='dot')
    except Exception as e:
        print('Note: Install system graphviz package to view the graph visualization natively.')
else:
    print('Cannot instantiate CausalModel: data is empty.')


In [ ]:
if not df.empty:
    estimand = identify_effect(model)
    print(estimand)


In [ ]:
if not df.empty:
    common_causes = [col for col in df.columns if col.startswith('U_pca_')]
    common_causes += ['U_dwell_mean', 'I_category', 'I_sentiment']
    ps = positivity_check(df, common_causes)


In [ ]:
if not df.empty:
    estimate_ipw = estimate_ate_ipw(model, estimand)
    print(f'ATE (IPW): {estimate_ipw.value:.4f}')
    estimate_lr = estimate_ate_linear(model, estimand)
    print(f'ATE (Linear Regression): {estimate_lr.value:.4f}')


In [ ]:
if not df.empty:
    if 'user_id' in df.columns and 'item_id' in df.columns:
        sample_df = df.head(10).copy()
        try:
            cdi_scores = compute_cdi_batch(model, estimand, sample_df)
            print('Sample CDI Scores (Proxy via ATE without EconML):')
            for k, v in list(cdi_scores.items())[:3]:
                print(f'> User {k[0]} -> Item {k[1]}: {v:.4f}')
        except Exception as e:
            print('Could not compute ITE:', e)
    else:
        print('Data missing user_id or item_id columns.')


In [ ]:
if not df.empty:
    print('--- Starting Refutations ---')
    results = run_refutations(model, estimand, estimate_ipw)
    print('\nRefutation checks completed.')
